# Lecture 11: Graph Neural Networks (GNN)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.linalg import eigh
from sklearn.preprocessing import normalize
from collections import defaultdict

np.random.seed(42)
print("Libraries loaded.")
print(f"NetworkX: {nx.__version__}")


## 1. What is a Graph? (Ch. 13.1)

A graph **G = (V, E)** consists of:
- **V**: Set of nodes — N nodes
- **E**: Set of edges — connections

Real-world graphs:
- Social networks (person = node, friendship = edge)
- Molecules (atom = node, chemical bond = edge)
- Citation networks (paper = node, citation = edge)
- Road networks (intersection = node, road = edge)

The **adjacency matrix A** is used for mathematical representation:
$$A_{mn} = 1 \\text{ if } m \\leftrightarrow n \\text{ connected, else } 0$$


In [ ]:
# Build and visualize a simple graph
G = nx.Graph()
G.add_nodes_from(range(6))
edges = [(0,1),(0,2),(1,2),(1,3),(2,4),(3,4),(3,5),(4,5)]
G.add_edges_from(edges)

# Adjacency matrix
A = nx.to_numpy_array(G, dtype=int)

# Node features (example: each node is 3-dimensional)
np.random.seed(7)
X = np.random.randn(6, 3).round(2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Graph visualization
pos = nx.spring_layout(G, seed=42)
node_colors = plt.cm.Set2(np.linspace(0, 1, 6))
nx.draw_networkx(G, pos, ax=axes[0],
                 node_color=node_colors, node_size=600,
                 font_color='white', font_weight='bold',
                 edge_color='gray', width=2)
axes[0].set_title("Graph G = (V, E)\n6 nodes, 8 edges", fontsize=12)
axes[0].axis('off')

# Adjacency matrix heatmap
im = axes[1].imshow(A, cmap='Blues', vmin=0, vmax=1)
for i in range(6):
    for j in range(6):
        axes[1].text(j, i, str(A[i,j]), ha='center', va='center',
                     fontsize=12, color='white' if A[i,j] else 'lightgray')
axes[1].set_title("Adjacency Matrix A\n(N×N binary matrix)", fontsize=12)
axes[1].set_xticks(range(6)); axes[1].set_yticks(range(6))
plt.colorbar(im, ax=axes[1])

# Node feature matrix
im2 = axes[2].imshow(X, cmap='RdBu_r', aspect='auto')
axes[2].set_title("Node Feature Matrix X\n(N×D, each row = one node)", fontsize=12)
axes[2].set_yticks(range(6)); axes[2].set_yticklabels([f"v{i}" for i in range(6)])
axes[2].set_xlabel("Feature dimension (D=3)")
plt.colorbar(im2, ax=axes[2])
for i in range(6):
    for j in range(3):
        axes[2].text(j, i, f"{X[i,j]:.1f}", ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig("fig_nb11_graph_repr.png", dpi=100, bbox_inches='tight')
plt.show()

print("Degree vector (neighbors per node):", A.sum(axis=1).astype(int))
print("Total edge count:", int(A.sum()//2))


## 2. Powers of the Adjacency Matrix (Ch. 13.2)

An important observation from the textbook:

> The entry at position $(m,n)$ of $A^L$ gives the number of **walks of length $L$** from $m$ to $n$.

This encodes connectivity information between nodes and explains why GNN layers can capture multi-hop neighborhoods.


In [ ]:
# Compute powers of A
A_float = A.astype(float)
A2 = A_float @ A_float   # 2-hop walks
A3 = A2 @ A_float        # 3-hop walks

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (mat, title) in zip(axes, [
    (A,              "A   (1-hop walk)"),
    (A2.astype(int), "A²  (2-hop walk)"),
    (A3.astype(int), "A³  (3-hop walk)"),
]):
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    for i in range(6):
        for j in range(6):
            ax.text(j, i, str(int(mat[i,j])), ha='center', va='center', fontsize=11,
                    color='white' if mat[i,j] > mat.max()*0.6 else 'black')
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels([f"v{i}" for i in range(6)])
    ax.set_yticklabels([f"v{i}" for i in range(6)])
    plt.colorbar(im, ax=ax)

plt.suptitle("Powers of the Adjacency Matrix: L-hop Connectivity", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_adj_powers.png", dpi=100, bbox_inches='tight')
plt.show()

# Reachability from v0
print("Starting from v0:")
print(f"  Reached in 1 hop: v{np.where(A[0]>0)[0].tolist()} → {int(A[0].sum())} walks")
print(f"  2-hop total walks: {int(A2[0].sum())}")
print(f"  3-hop total walks: {int(A3[0].sum())}")


## 3. GNN Tasks and Loss Functions (Ch. 13.3)

GNNs can solve three main task types:

| Task | Target | Example |
|---|---|---|
| **Graph classification** | Label for the whole graph | Molecular toxicity |
| **Node classification** | Label for each node | Community detection in social networks |
| **Link prediction** | Does an edge exist? | Recommendation system |

The loss function for each task is a standard classification/regression loss, applied at the node/edge/graph level.


In [ ]:
# Visualize the 3 task types
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pos_vis = nx.spring_layout(G, seed=42)

# --- Graph Classification ---
ax = axes[0]
nx.draw_networkx(G, pos_vis, ax=ax, node_color='steelblue', node_size=500,
                 font_color='white', edge_color='gray', width=2)
ax.set_title("Graph Classification\n→ Label for entire graph", fontsize=11)
ax.text(0.5, -0.12, "Example: Is this molecule toxic?",
        ha='center', transform=ax.transAxes, fontsize=9, color='gray')
ax.axis('off')

# --- Node Classification ---
ax = axes[1]
node_cls = [0, 0, 1, 1, 0, 1]  # 2 classes
colors_cls = ['crimson' if c else 'steelblue' for c in node_cls]
nx.draw_networkx(G, pos_vis, ax=ax, node_color=colors_cls, node_size=500,
                 font_color='white', edge_color='gray', width=2)
ax.set_title("Node Classification\n→ Label for each node", fontsize=11)
ax.text(0.5, -0.12, "Example: Red=Community A, Blue=Community B",
        ha='center', transform=ax.transAxes, fontsize=9, color='gray')
legend_elems = [mpatches.Patch(color='steelblue', label='Class 0'),
                mpatches.Patch(color='crimson',   label='Class 1')]
ax.legend(handles=legend_elems, loc='lower right', fontsize=8)
ax.axis('off')

# --- Link Prediction ---
ax = axes[2]
nx.draw_networkx(G, pos_vis, ax=ax, node_color='darkorange', node_size=500,
                 font_color='white', edge_color='gray', width=2)
x0, y0 = pos_vis[0]; x5, y5 = pos_vis[5]
ax.annotate("", xy=(x5, y5), xytext=(x0, y0),
            arrowprops=dict(arrowstyle="-", color='red', lw=2.5, linestyle='dashed'))
ax.text((x0+x5)/2+0.05, (y0+y5)/2+0.05, "?", fontsize=16, color='red', fontweight='bold')
ax.set_title("Link Prediction\n→ Does this edge exist?", fontsize=11)
ax.text(0.5, -0.12, "Example: Will these two people become friends?",
        ha='center', transform=ax.transAxes, fontsize=9, color='gray')
ax.axis('off')

plt.suptitle("GNN Task Types", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_nb11_tasks.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Graph Convolution Layer — Message Passing (Ch. 13.4)

The basic GCN layer update rule from the textbook:

$$H_{k+1} = a\\!\\left[\\beta_k \\mathbf{1}^T + \\Omega_k H_k (A + I)\\right]$$

- $H_k$: Node embedding matrix at layer $k$ (D×N)
- $A + I$: Adjacency + self-loop
- $\\Omega_k$: Learned weight matrix
- $\\beta_k$: Bias
- $a[\\cdot]$: ReLU activation

**Intuition:** Each node aggregates representations from its neighbors and applies a linear transformation.  
This is called **message passing**.


In [ ]:
def relu(x):
    return np.maximum(0, x)

def gcn_layer(H, A, W, b):
    '''
    Simple GCN layer.
    H: (N, D_in)  - node embeddings
    A: (N, N)     - adjacency matrix
    W: (D_in, D_out)
    b: (D_out,)
    Returns: (N, D_out)
    '''
    A_hat = A + np.eye(len(A))       # add self-loop
    return relu(A_hat @ H @ W + b)   # (N, D_out)

# Trace message passing step by step on a small example
N = 6
D_in, D_hidden = 3, 4
np.random.seed(1)
W1 = np.random.randn(D_in, D_hidden) * 0.3
b1 = np.zeros(D_hidden)

H0 = X.copy()           # Init: raw node features
H1 = gcn_layer(H0, A, W1, b1)   # After layer 1

print(f"H0 (input) shape:      {H0.shape}")
print(f"H1 (layer 1) shape:    {H1.shape}")
print()
print("Message passing steps for v1:")
print(f"  v1 neighbors: {list(G.neighbors(1))}")
print(f"  H0[v1]          = {H0[1].round(2)}")
agg_v1 = H0[[1] + list(G.neighbors(1))].sum(axis=0)
print(f"  Aggregated (v1+neighbors) = {agg_v1.round(2)}")
print(f"  H1[v1] = ReLU(aggr @ W + b) = {H1[1].round(3)}")

# 2-layer GCN
W2 = np.random.randn(D_hidden, 2) * 0.3
b2 = np.zeros(2)
H2 = gcn_layer(H1, A, W2, b2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (H, title) in zip(axes, [
    (H0, "H0: Raw features (D=3)"),
    (H1, "H1: After GCN layer 1 (D=4)"),
    (H2, "H2: After GCN layer 2 (D=2)"),
]):
    im = ax.imshow(H, cmap='RdBu_r', aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_yticks(range(N))
    ax.set_yticklabels([f"v{i}" for i in range(N)])
    ax.set_xlabel("Embedding dimension")
    plt.colorbar(im, ax=ax)

plt.suptitle("GCN Layers: Node Embeddings After Message Passing", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_gcn_layers.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. K-Hop Neighborhood and Receptive Field (Ch. 13.7)

Each GCN layer aggregates information from a node's **1-hop neighborhood**.  
After K layers, each node sees its **K-hop neighborhood**.

This is the graph analogue of the **receptive field** concept in CNNs.


In [ ]:
def get_k_hop_neighborhood(G, node, k):
    '''Returns the set of nodes in the k-hop neighborhood of a given node.'''
    neighbors = {node}
    frontier = {node}
    for _ in range(k):
        next_frontier = set()
        for n in frontier:
            next_frontier.update(G.neighbors(n))
        next_frontier -= neighbors
        neighbors.update(next_frontier)
        frontier = next_frontier
    return neighbors

target_node = 0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k in zip(axes, [1, 2, 3]):
    hop_nodes = get_k_hop_neighborhood(G, target_node, k)

    colors = []
    for n in G.nodes():
        if n == target_node:
            colors.append('crimson')
        elif n in hop_nodes:
            colors.append('steelblue')
        else:
            colors.append('lightgray')

    pos_k = nx.spring_layout(G, seed=42)
    nx.draw_networkx(G, pos_k, ax=ax, node_color=colors, node_size=600,
                     font_color='white', edge_color='gray', width=2)
    ax.set_title(f"{k}-Hop Neighborhood\nReceptive field of v{target_node} (red)", fontsize=11)

    legend_elems = [mpatches.Patch(color='crimson',   label=f'v{target_node} (target)'),
                    mpatches.Patch(color='steelblue', label=f'{k}-hop neighbors ({len(hop_nodes)-1})'),
                    mpatches.Patch(color='lightgray', label='Outside')]
    ax.legend(handles=legend_elems, loc='lower right', fontsize=7)
    ax.axis('off')

plt.suptitle("K-Hop Neighborhood = GCN Receptive Field", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_receptive_field.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Kipf Normalization (Ch. 13.8.3)

In standard aggregation, high-degree nodes dominate low-degree neighbors.  
Kipf & Welling (2017) propose the normalized adjacency matrix:

$$\\tilde{A} = D^{-1/2}(A+I)D^{-1/2}$$

where $D$ is the degree matrix (diagonal). This normalization:
- Balances information flow by degree
- Improves gradient stability
- Yields large practical gains


In [ ]:
def kipf_normalize(A):
    '''Symmetrically normalized Kipf adjacency matrix.'''
    A_hat = A + np.eye(len(A))           # add self-loop
    D = np.diag(A_hat.sum(axis=1))       # degree matrix
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))
    return D_inv_sqrt @ A_hat @ D_inv_sqrt

def gcn_kipf_layer(H, A_norm, W, b):
    return relu(A_norm @ H @ W + b)

A_norm = kipf_normalize(A)

# Compare normalized vs un-normalized
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(A + np.eye(6), cmap='Blues', vmin=0, vmax=1)
axes[0].set_title("A + I (unnormalized)", fontsize=11)
plt.colorbar(im0, ax=axes[0])
for i in range(6):
    for j in range(6):
        v = int((A + np.eye(6))[i, j])
        axes[0].text(j, i, str(v), ha='center', va='center', fontsize=11,
                     color='white' if v else 'lightgray')

im1 = axes[1].imshow(A_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title("Ã = D⁻¹²(A+I)D⁻¹²\n(Kipf normalization)", fontsize=11)
plt.colorbar(im1, ax=axes[1])
for i in range(6):
    for j in range(6):
        v = A_norm[i, j]
        axes[1].text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=9,
                     color='white' if v > 0.4 else 'black')

degrees = A.sum(axis=1)
axes[2].bar(range(6), degrees, color='steelblue', edgecolor='white')
axes[2].bar(range(6), 1/np.sqrt(degrees), color='crimson', alpha=0.7, label='Weight 1/√d')
axes[2].set_title("Kipf: High-degree nodes\nreceive lower weight", fontsize=11)
axes[2].set_xlabel("Node"); axes[2].set_ylabel("Value")
axes[2].set_xticks(range(6))
axes[2].set_xticklabels([f"v{i}\n(d={int(degrees[i])})" for i in range(6)])
axes[2].legend(); axes[2].grid(axis='y', alpha=0.3)
for ax in axes[:2]:
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels([f"v{i}" for i in range(6)])
    ax.set_yticklabels([f"v{i}" for i in range(6)])

plt.suptitle("Kipf Normalization: Degree-Weighted Message Passing", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_kipf.png", dpi=100, bbox_inches='tight')
plt.show()

print("Row sums (normalization check):")
print(A_norm.sum(axis=1).round(3))


## 7. Node Classification: Full GCN Training (Ch. 13.7)

We will perform real node classification using the Karate Club dataset (Zachary, 1977):
- 34 nodes (people), 78 edges (interactions)
- 2 classes: membership in one of two clubs (ground-truth split)
- Only **some node labels** are used during training (transductive setting)


In [ ]:
def softmax(z, axis=-1):
    e = np.exp(z - z.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

# Zachary Karate Club graph
G_kc = nx.karate_club_graph()
N_kc = G_kc.number_of_nodes()

# Ground-truth labels
labels_kc = np.array([1 if G_kc.nodes[n]['club'] == 'Officer' else 0
                       for n in G_kc.nodes()])

A_kc = nx.to_numpy_array(G_kc)
A_kc_norm = kipf_normalize(A_kc)

# Initial features: normalized degree + small random projection
np.random.seed(5)
degrees_kc = A_kc.sum(axis=1, keepdims=True)
X_proj     = np.random.randn(N_kc, 8) * 0.5
X_kc = np.concatenate([degrees_kc / degrees_kc.max(), X_proj], axis=1)
D_in_kc = X_kc.shape[1]   # 9

# 2-layer GCN — manual SGD (no PyTorch)
np.random.seed(3)
D_h = 8
W1_kc = np.random.randn(D_in_kc, D_h) * 0.1
W2_kc = np.random.randn(D_h, 2) * 0.1

def softmax_2d(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def gcn_forward(X, A_n, W1, W2):
    H1 = relu(A_n @ X @ W1)
    H2 = A_n @ H1 @ W2
    return softmax_2d(H2), H1

def ce_loss(probs, y_idx):
    return -np.mean(np.log(probs[np.arange(len(y_idx)), y_idx] + 1e-9))

# Training mask: 5 nodes per class
train_mask = np.zeros(N_kc, dtype=bool)
for cls in [0, 1]:
    idxs = np.where(labels_kc == cls)[0][:5]
    train_mask[idxs] = True

lr = 0.1
losses, accs_tr, accs_te = [], [], []

for epoch in range(200):
    probs, H1 = gcn_forward(X_kc, A_kc_norm, W1_kc, W2_kc)
    loss = ce_loss(probs[train_mask], labels_kc[train_mask])
    losses.append(loss)

    # Analytic gradient (softmax + cross-entropy)
    N_tr  = train_mask.sum()
    delta = probs.copy()
    delta[train_mask, labels_kc[train_mask]] -= 1
    delta[~train_mask] = 0
    delta /= N_tr

    # Layer 2 gradient: H2 = A @ H1 @ W2
    AH1 = A_kc_norm @ H1
    dW2 = AH1.T @ delta

    # Backprop to layer 1
    dAH1 = delta @ W2_kc.T
    dH1  = A_kc_norm.T @ dAH1
    dZ1  = dH1 * (H1 > 0)          # ReLU derivative
    dW1  = (A_kc_norm @ X_kc).T @ dZ1

    W1_kc -= lr * dW1
    W2_kc -= lr * dW2

    preds = probs.argmax(axis=1)
    accs_tr.append((preds[train_mask]  == labels_kc[train_mask]).mean())
    accs_te.append((preds[~train_mask] == labels_kc[~train_mask]).mean())

print(f"Final train accuracy: {accs_tr[-1]:.3f}")
print(f"Final test accuracy:  {accs_te[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(losses,  color='steelblue', lw=2, label="Training loss (CE)")
axes[0].plot(accs_tr, color='crimson',    lw=2, label="Train accuracy")
axes[0].plot(accs_te, color='darkorange', lw=2, label="Test accuracy")
axes[0].set_title("2-Layer GCN: Karate Club Node Classification", fontsize=12)
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

probs_final, _ = gcn_forward(X_kc, A_kc_norm, W1_kc, W2_kc)
preds_final = probs_final.argmax(axis=1)

pos_kc = nx.spring_layout(G_kc, seed=0)
node_cols = []
for n in G_kc.nodes():
    pred = preds_final[n]; true = labels_kc[n]
    if pred == true:
        node_cols.append('steelblue' if true == 0 else 'crimson')
    else:
        node_cols.append('gold')

nx.draw_networkx(G_kc, pos_kc, ax=axes[1], node_color=node_cols,
                 node_size=220, font_size=7, font_color='white',
                 edge_color='lightgray', width=0.8, with_labels=True)
train_nodes = list(np.where(train_mask)[0])
nx.draw_networkx_nodes(G_kc, pos_kc, nodelist=train_nodes, ax=axes[1],
                       node_color=[node_cols[n] for n in train_nodes],
                       node_size=350, linewidths=2, edgecolors='black')

legend_elems = [mpatches.Patch(color='steelblue', label='Class 0 (correct)'),
                mpatches.Patch(color='crimson',   label='Class 1 (correct)'),
                mpatches.Patch(color='gold',      label='Misclassified')]
axes[1].legend(handles=legend_elems, loc='lower left', fontsize=7)
axes[1].set_title(f"GCN Predictions (test acc={accs_te[-1]:.2f})", fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.savefig("fig_nb11_node_clf.png", dpi=100, bbox_inches='tight')
plt.show()


## 8. Graph Attention Network (GAT) (Ch. 13.8.5)

In GCN, neighborhood weights are **fixed** (degree-based).  
In GAT, weights are **data-dependent** (learned attention):

$$s_{mn} = a\\!\\left[\\phi^T \\begin{pmatrix} h_m' \\\\ h_n' \\end{pmatrix}\\right]$$

$$H_{k+1} = a\\!\\left[H_k' \\cdot \\text{Softmask}(S, A+I)\\right]$$

Can be seen as the combination of Transformer self-attention with graph structure.


In [ ]:
def gat_layer(H, A, W, a_vec, activation=relu):
    '''
    Simplified GAT layer.
    H: (N, D_in)
    A: (N, N) adjacency
    W: (D_in, D_out) linear transform
    a_vec: (2*D_out,) attention vector
    '''
    N, D_in = H.shape
    D_out = W.shape[1]
    H_prime = H @ W                      # (N, D_out)

    # Attention scores
    S = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if A[i,j] > 0 or i == j:     # only neighbors + self
                concat = np.concatenate([H_prime[i], H_prime[j]])
                S[i,j] = np.dot(a_vec, concat)

    # Softmax (only for neighbors + self)
    mask = (A + np.eye(N)) > 0
    S_masked = np.where(mask, S, -1e9)
    exp_S = np.exp(S_masked - S_masked.max(axis=1, keepdims=True))
    attn_weights = exp_S / (exp_S.sum(axis=1, keepdims=True) + 1e-9)
    attn_weights = attn_weights * mask

    H_out = activation(attn_weights @ H_prime)
    return H_out, attn_weights

np.random.seed(5)
D_out_gat = 4
W_gat = np.random.randn(D_in, D_out_gat) * 0.3
a_gat  = np.random.randn(2 * D_out_gat) * 0.3

H_gat, attn = gat_layer(X, A, W_gat, a_gat)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(attn, cmap='Blues', vmin=0, vmax=attn.max())
axes[0].set_title("GAT Attention Weights\n(Row = query node, Column = key node)", fontsize=11)
axes[0].set_xticks(range(6)); axes[0].set_yticks(range(6))
axes[0].set_xticklabels([f"v{i}" for i in range(6)])
axes[0].set_yticklabels([f"v{i}" for i in range(6)])
for i in range(6):
    for j in range(6):
        axes[0].text(j, i, f"{attn[i,j]:.2f}", ha='center', va='center', fontsize=9,
                     color='white' if attn[i,j] > 0.3 else 'black')
plt.colorbar(im, ax=axes[0])

# Visualize attention weights as edge thickness on the graph
ax = axes[1]
pos_vis2 = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos_vis2, ax=ax, node_color='steelblue', node_size=500)
nx.draw_networkx_labels(G, pos_vis2, ax=ax, font_color='white', font_weight='bold')

for u, v in G.edges():
    w = (attn[u,v] + attn[v,u]) / 2
    nx.draw_networkx_edges(G, pos_vis2, edgelist=[(u,v)], ax=ax,
                           width=w*12, alpha=0.6, edge_color='darkorange')

ax.set_title("Attention Weights on Graph\n(Edge thickness = attention)", fontsize=11)
ax.axis('off')

plt.suptitle("Graph Attention Network (GAT): GCN vs Learned Attention", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_gat.png", dpi=100, bbox_inches='tight')
plt.show()


## 9. Real-World Example: Molecule Classification (Ch. 13.5)

One of the most successful application domains of GNNs is **molecular chemistry**:
- Atom = Node, Chemical bond = Edge
- Goal: Predict whether a molecule is toxic or not

Below we represent a few simple molecules using a GNN and classify them.


In [ ]:
# Simple molecules: Ethanol (C2H5OH) and Acetic Acid (CH3COOH) -like structures
# Using only atom type and bond type features

# Atom encoding: C=0, O=1, N=2, H=3 (one-hot)
def atom_onehot(atom_type, n_types=4):
    v = np.zeros(n_types)
    v[atom_type] = 1.0
    return v

def make_molecule_graph(atom_types, bond_list):
    n = len(atom_types)
    A = np.zeros((n, n))
    for i, j in bond_list:
        A[i,j] = A[j,i] = 1
    X = np.array([atom_onehot(t) for t in atom_types])
    return A, X

molecules = [
    # (atom_types, bonds, label, name)
    ([0, 0, 1], [(0,1),(1,2)],       0, "Ethanol-like\n(C-C-O)"),
    ([0, 1, 1], [(0,1),(0,2)],       0, "Dimethyl ether\n(O-C-O)"),
    ([0, 0, 0, 1], [(0,1),(1,2),(2,3),(0,3)], 1, "Cyclic mol.\n(ring)"),
    ([0, 2, 0, 1], [(0,1),(1,2),(2,3),(0,3)], 1, "N-containing ring"),
    ([0, 0, 1, 1], [(0,1),(1,2),(1,3)],       0, "Branched chain"),
    ([0, 0, 0, 2], [(0,1),(1,2),(2,3),(0,2)], 1, "Closed N-ring"),
]

np.random.seed(8)
D_mol = 4    # one-hot atom
D_h1  = 8
D_h2  = 4
W_m1 = np.random.randn(D_mol, D_h1) * 0.3
W_m2 = np.random.randn(D_h1, D_h2) * 0.3
W_clf = np.random.randn(D_h2, 2) * 0.3

def molecule_forward(A_m, X_m, W1, W2, Wc):
    A_n = kipf_normalize(A_m)
    H1 = relu(A_n @ X_m @ W1)
    H2 = relu(A_n @ H1 @ W2)
    # Graph-level: global mean pooling over nodes
    g = H2.mean(axis=0, keepdims=True)    # (1, D_h2)
    logits = g @ Wc                        # (1, 2)
    probs = np.exp(logits) / np.exp(logits).sum()
    return probs[0], H2

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (atom_types, bonds, label, name) in zip(axes.flat, molecules):
    A_m, X_m = make_molecule_graph(atom_types, bonds)
    probs_mol, H2_mol = molecule_forward(A_m, X_m, W_m1, W_m2, W_clf)
    pred_label = probs_mol.argmax()

    G_mol = nx.from_numpy_array(A_m)
    pos_mol = nx.spring_layout(G_mol, seed=42)
    atom_names  = {0:'C', 1:'O', 2:'N', 3:'H'}
    atom_colors = {0:'gray', 1:'crimson', 2:'steelblue', 3:'white'}
    node_colors_mol = [atom_colors[t] for t in atom_types]
    labels_mol = {i: atom_names[t] for i, t in enumerate(atom_types)}

    nx.draw_networkx(G_mol, pos_mol, ax=ax,
                     node_color=node_colors_mol, node_size=700,
                     labels=labels_mol, font_color='white', font_weight='bold',
                     edge_color='dimgray', width=3)
    correct = "✓" if pred_label == label else "✗"
    ax.set_title(f"{name}\nTrue: {'Toxic' if label else 'Normal'} | "
                 f"Pred: {'Toxic' if pred_label else 'Normal'} {correct}",
                 fontsize=9)
    ax.axis('off')

plt.suptitle("Molecule Classification with GNN (random weights)", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb11_molecules.png", dpi=100, bbox_inches='tight')
plt.show()


## 10. GCN, CNN, and Transformer Comparison (Ch. 13.8.5)

| Feature | CNN | Transformer | GNN |
|---|---|---|---|
| Input structure | Regular grid | Sequence | General graph |
| Neighborhood | Fixed (sliding window) | Full (all-to-all) | Graph topology |
| Weight sharing | All positions | All positions | All nodes |
| Permutation | Translation equivariant | Requires positional encoding | Permutation equivariant |
| Typical data | Images, audio | Text, images | Molecules, social networks |

All these architectures can be unified under the **message passing** framework.


## 11. Summary

| Concept | Description |
|---|---|
| **Adjacency matrix A** | N×N binary matrix encoding graph connectivity |
| **Node embedding H** | Representation vector for each node (D-dimensional) |
| **Message passing** | Each node aggregates information from its neighbors |
| **GCN layer** | $H_{k+1} = a[\\Omega H_k (A+I) + \\beta]$ |
| **Kipf normalization** | $\\tilde{A} = D^{-1/2}(A+I)D^{-1/2}$ — balances degree differences |
| **GAT** | Neighborhood weights are learned from data (attention-based) |
| **K-hop neighborhood** | K layers → K-hop receptive field |
| **Graph classification** | Aggregate nodes via global mean/max pooling |
| **Permutation equivariance** | Output is invariant to node ordering |

---
**GNN Applications:**
- Molecular property prediction (drug discovery)
- Social network analysis (community detection)
- Recommender systems (link prediction)
- Traffic forecasting (road network analysis)
- Protein structure prediction (AlphaFold-like)
